In [ ]:
import asyncio
import os
import pandas as pd
from tabulate import tabulate
from fastmcp import Client
from fastmcp.client.transports import StreamableHttpTransport
from langchain_core.tools import StructuredTool
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from typing import TypedDict
import json
import re

# -----------------------------
# Define state schema (optional)
# -----------------------------
class AgentState(TypedDict):
    user_input: str
    agent_response: str

# -----------------------------
# Initialize LLM
# -----------------------------
model = init_chat_model(
    "ollama:qwen2.5:3b",
    model_kwargs={"temperature": 0, "num_predict": 300},
    base_url="http://ollama:11434"
)

# -----------------------------
# Sync wrapper for MCP tools
# -----------------------------
def make_sync_wrapper(tool_name, client):
    def sync_wrapper(sql: str) -> str:
        try:
            sql = sql.rstrip(";").strip()

            loop = asyncio.get_event_loop()
            TIMEOUT = int(os.getenv("MCP_CLIENT_TIMEOUT", 60))  # ⬅ configurable timeout (default 60s)
            coro = client.call_tool(tool_name, {"sql": sql})
            result = loop.run_until_complete(asyncio.wait_for(coro, timeout=TIMEOUT))

            # Return the raw JSON response for proper parsing
            if hasattr(result, 'data') and result.data:
                return json.dumps(result.data)
            elif hasattr(result, "structuredContent") and result.structuredContent:
                return json.dumps(result.structuredContent)
            elif hasattr(result, "content") and result.content:
                return result.content[0].text if isinstance(result.content, list) else str(result.content)
            else:
                return str(result)
        except asyncio.TimeoutError:
            return f"❌ Timeout calling {tool_name}"
        except Exception as e:
            return f"❌ Error calling {tool_name}: {str(e)}"

    return sync_wrapper

def extract_and_display_table(result):
    """Extract table data from agent result and display it in tabulated format"""
    
    print(f"🔍 DEBUG: extract_and_display_table called")
    print(f"🔍 DEBUG: result type: {type(result)}")
    
    # Helper function to extract JSON from text
    def extract_json_from_text(text):
        if not text:
            return None
        
        print(f"🔍 DEBUG: Trying to parse text of length {len(text)}")
        print(f"🔍 DEBUG: Text preview: {text[:200]}...")
        
        # Try direct JSON parsing first
        try:
            parsed = json.loads(text)
            print(f"🔍 DEBUG: Successfully parsed JSON")
            return parsed
        except Exception as e:
            print(f"🔍 DEBUG: Direct JSON parse failed: {e}")
            pass
        
        # Look for JSON pattern in text
        json_pattern = r'\{[^{}]*"success"[^{}]*\}'
        matches = re.findall(json_pattern, text)
        for match in matches:
            try:
                return json.loads(match)
            except:
                continue
        
        # Look for more complex nested JSON
        brace_count = 0
        start_idx = -1
        for i, char in enumerate(text):
            if char == '{':
                if brace_count == 0:
                    start_idx = i
                brace_count += 1
            elif char == '}':
                brace_count -= 1
                if brace_count == 0 and start_idx != -1:
                    try:
                        json_str = text[start_idx:i+1]
                        parsed = json.loads(json_str)
                        if 'success' in parsed:
                            return parsed
                    except:
                        pass
        
        return None
    
    # Get messages from result
    messages = []
    if hasattr(result, "messages"):
        messages = result.messages
    elif isinstance(result, dict) and "messages" in result:
        messages = result["messages"]
    
    print(f"🔍 DEBUG: Found {len(messages)} messages")
    
    # Look through all messages for tool responses with JSON data
    for i, msg in enumerate(messages):
        msg_type = getattr(msg, 'type', None) or str(type(msg).__name__).lower()
        print(f"🔍 DEBUG: Message {i}: type={msg_type}")
        
        if 'tool' in msg_type:
            print(f"🔍 DEBUG: Found tool message")
            content = getattr(msg, 'content', None)
            print(f"🔍 DEBUG: Content type: {type(content)}")
            print(f"🔍 DEBUG: Content: {content}")
            
            if content:
                # Handle different content formats
                content_text = None
                if isinstance(content, str):
                    content_text = content
                elif isinstance(content, list) and content:
                    if isinstance(content[0], dict) and "text" in content[0]:
                        content_text = content[0]["text"]
                    else:
                        content_text = str(content[0])
                
                if content_text:
                    table_data = extract_json_from_text(content_text)
                    if table_data and table_data.get("success"):
                        print(f"🔍 DEBUG: Successfully found table data")
                        display_table_from_json(table_data)
                        return True
                    else:
                        print(f"🔍 DEBUG: No valid table data found")
    
    print(f"🔍 DEBUG: No table found in any message")
    return False

def display_table_from_json(data):
    """Display table data in tabulated format"""
    preview_rows = data.get("preview_rows", [])
    preview_cols = data.get("preview_columns", [])
    total_rows = data.get("total_rows", len(preview_rows))
    total_cols = data.get("total_columns", len(preview_cols))
    
    if preview_rows and preview_cols:
        df = pd.DataFrame(preview_rows, columns=preview_cols)
        df.index = df.index + 1
        
        # Add "..." if rows or columns truncated
        if total_rows > len(preview_rows):
            df.loc["..."] = ["..."] * len(df.columns)
        if total_cols > len(preview_cols):
            df["..."] = "..."
        
        print("\n📊 Query Result (preview):")
        print(tabulate(df, headers="keys", tablefmt="pretty"))
        print(f"\nShowing preview: {len(preview_rows)} rows × {len(preview_cols)} columns "
              f"(full result: {total_rows} rows × {total_cols} columns)")
        
        # Show debug info if available
        debug_info = data.get("_debug_info", {})
        if debug_info:
            print(f"🔍 Debug: Query returned {debug_info.get('query_returned_rows', '?')} rows, "
                  f"True total: {debug_info.get('true_total_rows', '?')} rows")
    else:
        print("ℹ️ Query executed but returned no rows.")

# -----------------------------
# Define system prompt
# -----------------------------
system_prompt = (
    "You are a SQL assistant. Always call the available tools. `query_db` tool to run queries. "
    "Do not guess results. Always show the executed SQL query."
)

# -----------------------------
# Main async function
# -----------------------------
async def main():
    transport = StreamableHttpTransport("http://mcp_server:8000/mcp")
    async with Client(transport=transport) as client:

        tools_list = await client.list_tools()
        tools = []

        for tool in tools_list:
            tool_name = getattr(tool, "name", None)
            description = getattr(tool, "description", f"MCP tool {tool_name}")

            tool_func = make_sync_wrapper(tool_name, client)
            structured_tool = StructuredTool.from_function(
                func=tool_func,
                name=tool_name,
                description=description
            )
            tools.append(structured_tool)

        print("\n🔧 Available MCP Tools:")
        for t in tools:
            print(f"- {t.name}: {t.description}")

        agent_llm = create_react_agent(model=model, tools=tools)

        print("\n🤖 Agent ready! Ask SQL questions.")
        print("Type 'exit', 'quit', or 'q' to stop.")

        # -----------------------------
        # Conversation loop
        # -----------------------------
        conversation = [("system", system_prompt)]  # system prompt added here

        while True:
            try:
                user_input = input("\nEnter your SQL query: ").strip()
                if user_input.lower() in ("exit", "quit", "q"):
                    print("👋 Goodbye!")
                    break
                if not user_input:
                    continue

                print("🤔 Thinking...")

                # Prepend the system prompt to the conversation
                conversation.append(("user", user_input))
                result = await agent_llm.ainvoke({"messages": conversation})

                # Extract and show the SQL query the agent used
                messages = getattr(result, "messages", [])
                
                for msg in messages:
                    msg_type = getattr(msg, 'type', None) or str(type(msg).__name__).lower()
                    
                    if 'ai' in msg_type and hasattr(msg, 'content'):
                        content = msg.content
                        if isinstance(content, str) and ('SELECT' in content.upper() or 'query_db' in content):
                            print(f"\n🤖 Agent generated query:")
                            lines = content.split('\n')
                            for line in lines:
                                if 'SELECT' in line.upper() or 'query_db' in line:
                                    print(f"   {line.strip()}")
                            break
                
                for msg in messages:
                    msg_type = getattr(msg, 'type', None) or str(type(msg).__name__).lower()
                    if 'tool' in msg_type and hasattr(msg, 'name') and getattr(msg, 'name') == 'query_db':
                        args = getattr(msg, 'args', {}) or getattr(msg, 'tool_input', {})
                        if args and 'sql' in args:
                            print(f"\n🔍 SQL executed: {args['sql']}")

                table_displayed = extract_and_display_table(result)
                
                messages = getattr(result, "messages", [])
                
                final_response = None
                for msg in reversed(messages):
                    msg_type = getattr(msg, 'type', None) or str(type(msg).__name__).lower()
                    if 'ai' in msg_type and hasattr(msg, 'content') and msg.content:
                        final_response = msg.content
                        break
                
                if final_response:
                    print("\n🤖 Agent Response:")
                    print(final_response)
                elif not table_displayed:
                    print("\n🤖 No response available")

            except KeyboardInterrupt:
                print("\n👋 Goodbye!")
                break
            except Exception as e:
                print(f"\n❌ Unexpected error: {e}")
                import traceback
                print("Debug traceback:")
                traceback.print_exc()


# -----------------------------
# Run main
# -----------------------------
if __name__ == "__main__":
    try:
        asyncio.run(main())
    except RuntimeError as e:
        if "asyncio.run() cannot be called" in str(e):
            import nest_asyncio
            nest_asyncio.apply()
            loop = asyncio.get_event_loop()
            loop.run_until_complete(main())
        else:
            raise


🔧 Available MCP Tools:
- query_db: Execute a SELECT SQL query against PostgreSQL.
Returns:
  - preview (5 rows x 5 cols)
  - true total rows (ignoring LIMIT)
  - total columns

🤖 Agent ready! Ask SQL questions.
Type 'exit', 'quit', or 'q' to stop.



Enter your SQL query:  show me the content of departments table


🤔 Thinking...
🔍 DEBUG: extract_and_display_table called
🔍 DEBUG: result type: <class 'dict'>
🔍 DEBUG: Found 5 messages
🔍 DEBUG: Message 0: type=system
🔍 DEBUG: Message 1: type=human
🔍 DEBUG: Message 2: type=ai
🔍 DEBUG: Message 3: type=tool
🔍 DEBUG: Found tool message
🔍 DEBUG: Content type: <class 'str'>
🔍 DEBUG: Content: {"success": true, "preview_rows": [[1, "Department 1", 38, "Location 7", "365812.04"], [2, "Department 2", 48, "Location 4", "202327.53"], [3, "Department 3", 43, "Location 4", "853790.84"], [4, "Department 4", 21, "Location 4", "19482.89"], [5, "Department 5", 49, "Location 8", "619989.13"]], "preview_columns": ["department_id", "department_name", "manager_id", "location", "budget"], "total_rows": 50, "total_columns": 7, "duration_seconds": 0.010059, "_debug_info": {"original_query": "SELECT * FROM departments", "quoted_query": "SELECT * FROM \"departments\"", "base_query": "SELECT * FROM \"departments\"", "query_returned_rows": 50, "true_total_rows": 50}}
🔍 DEBUG: Tr


Enter your SQL query:  show me all contents of employees table


🤔 Thinking...
🔍 DEBUG: extract_and_display_table called
🔍 DEBUG: result type: <class 'dict'>
🔍 DEBUG: Found 7 messages
🔍 DEBUG: Message 0: type=system
🔍 DEBUG: Message 1: type=human
🔍 DEBUG: Message 2: type=human
🔍 DEBUG: Message 3: type=ai
🔍 DEBUG: Message 4: type=tool
🔍 DEBUG: Found tool message
🔍 DEBUG: Content type: <class 'str'>
🔍 DEBUG: Content: {"success": true, "preview_rows": [[1, "Department 1", 38, "Location 7", "365812.04"], [2, "Department 2", 48, "Location 4", "202327.53"], [3, "Department 3", 43, "Location 4", "853790.84"], [4, "Department 4", 21, "Location 4", "19482.89"], [5, "Department 5", 49, "Location 8", "619989.13"]], "preview_columns": ["department_id", "department_name", "manager_id", "location", "budget"], "total_rows": 50, "total_columns": 7, "duration_seconds": 0.009448, "_debug_info": {"original_query": "SELECT * FROM departments LIMIT 5", "quoted_query": "SELECT * FROM \"departments\" LIMIT 5", "base_query": "SELECT * FROM \"departments\"", "query_returned